# Al–Air Paste Battery Model — Getting Started

This notebook walks through the core model in 5 steps:
1. Single cell evaluation
2. Polarisation curve
3. Parameter sweep
4. Alloy optimisation
5. Calibration with your own data

**Standard conditions** (literature baseline): 25°C · 4M KOH · 53 vol% Al · 100µm particles

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from al_air_model import cell_model, BASE_CONFIG

print('BASE_CONFIG:', BASE_CONFIG)

BASE_CONFIG: {'d_um': 180, 'c_KOH': 4.0, 'vf_pct': 53, 'T_C': 25, 'inh_pct': 0}


## 1. Single cell evaluation
Run the physics model at a single operating point.

In [ ]:
r = cell_model(**BASE_CONFIG, j_mA_cm2=50)

print(f"Cell voltage:        {r['V_cell']:.3f} V")
print(f"OCV:                 {r['E_ocv']:.3f} V")
print(f"Energy density:      {r['ed_Wh_kg_paste']:.0f} Wh/kg paste")
print(f"Energy (system):     {r['ed_Wh_kg_system']:.0f} Wh/kg  (incl. ×0.612 engineering)")
print(f"Net useful energy:   {r['net_useful_ed']:.0f} Wh/kg")
print(f"Power density:       {r['pd_W_kg_paste']:.0f} W/kg")
print(f"Corrosion loss:      {r['parasitic_pct']:.2f} %")
print(f"Al utilisation:      {r['utilisation_pct']:.1f} %")
print(f"Dominant mechanism:  {r['dominant_mechanism']}")
print()
print("Overpotential budget:")
print(f"  η anode (BV):      {r['eta_anode_V']*1000:.1f} mV")
print(f"  η cathode (ORR):   {r['eta_cathode_V']*1000:.1f} mV")
print(f"  η ohmic:           {r['eta_ohmic_V']*1000:.1f} mV")
print(f"  η mass transport:  {r['eta_mass_trans_V']*1000:.1f} mV")

## 2. Polarisation curve
Sweep current density 1–70 mA/cm² and plot voltage + overpotential breakdown.

In [ ]:
j_vals = np.linspace(1, 70, 60)
results = [cell_model(**BASE_CONFIG, j_mA_cm2=j) for j in j_vals]

V      = [r['V_cell'] for r in results]
eta_bv = [(r['eta_anode_V'] + r['eta_cathode_V'])*1000 for r in results]
eta_oh = [r['eta_ohmic_V']*1000 for r in results]
eta_mt = [r['eta_mass_trans_V']*1000 for r in results]
ed     = [r['ed_Wh_kg_paste'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(j_vals, V, 'g-', linewidth=2, label='V cell')
axes[0].axhline(results[0]['E_ocv'], color='gray', linestyle='--', alpha=0.5, label=f'OCV={results[0]["E_ocv"]:.2f}V')
axes[0].set_xlabel('Current density (mA/cm²)')
axes[0].set_ylabel('Cell voltage (V)')
axes[0].set_title('Polarisation curve — 25°C, 4M KOH, 100µm')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].stackplot(j_vals, eta_bv, eta_oh, eta_mt,
                  labels=['η Butler-Volmer', 'η ohmic', 'η mass transport'],
                  colors=['#fb923c', '#fbbf24', '#38bdf8'], alpha=0.8)
axes[1].set_xlabel('Current density (mA/cm²)')
axes[1].set_ylabel('Overpotential (mV)')
axes[1].set_title('Loss breakdown')
axes[1].legend(loc='upper left')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('polarisation_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: polarisation_curve.png')

## 3. Parameter sweep — the particle size finding
The model predicts an optimal particle size at ~187 µm for paste cells.
This is a paste-specific finding — solid-plate models don't show this.

In [ ]:
d_vals = np.linspace(5, 500, 60)
net_energy = []
corrosion  = []

for d in d_vals:
    r = cell_model(d_um=d, c_KOH=4.0, vf_pct=53, T_C=25, inh_pct=0, j_mA_cm2=50)
    net_energy.append(r['net_useful_ed'])
    corrosion.append(r['parasitic_pct'])

# Find peak
peak_idx = np.argmax(net_energy)
peak_d   = d_vals[peak_idx]
peak_net = net_energy[peak_idx]
print(f'Peak net energy: {peak_net:.0f} Wh/kg at d = {peak_d:.0f} µm')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(d_vals, net_energy, 'b-', linewidth=2, label='Net useful energy')
ax1.axvline(peak_d, color='green', linestyle='--', alpha=0.7, label=f'Optimum: {peak_d:.0f} µm at {BASE_CONFIG["T_C"]}°C')
ax1.fill_between(d_vals, net_energy, alpha=0.1, color='blue')
ax1.set_xlabel('Particle diameter (µm)')
ax1.set_ylabel('Net energy density (Wh/kg)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2.plot(d_vals, corrosion, 'r--', linewidth=1.5, alpha=0.7, label='Corrosion %')
ax2.set_ylabel('Corrosion loss (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
ax1.set_title('Net energy vs particle size — paste-specific optimum (shifts with temperature)')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('particle_size_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Alloy optimisation
Find the best Al alloy composition for your operating conditions.

In [ ]:
from al_air_alloy import optimize_alloy

results, pareto = optimize_alloy(
    BASE_CONFIG,
    additives=['Mg', 'In', 'Sn', 'Zn', 'Ga'],
    goal='balanced',
    n_samples=2000,
    j=50
)

print(f'Valid configs: {len(results):,}')
print(f'Pareto front:  {len(pareto)} configs')
print()
print('Top 5 balanced compositions:')
print(f'{"#":>3}  {"Composition":^45}  {"Net Wh/kg":>10}  {"Corr%":>6}  Synergy')
print('-' * 80)
for i, r in enumerate(results[:5]):
    comp = '+'.join(f"{k}:{v*100:.1f}%" for k,v in r['comp'].items() if k!='Al' and v>0.001)
    print(f'{i+1:>3}  {comp:<45}  {r["net_ed"]:>10.0f}  {r["corr"]:>6.2f}  {"Yes" if r["synergy"] else "No"}')

In [ ]:
# Plot design space
all_net  = [r['net_ed']  for r in results]
all_corr = [r['corr']    for r in results]
all_V    = [r['voltage'] for r in results]
par_net  = [r['net_ed']  for r in pareto]
par_corr = [r['corr']    for r in pareto]

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(all_corr, all_net, c=all_V, cmap='viridis', alpha=0.4, s=15)
ax.scatter(par_corr, par_net, color='orange', s=60, zorder=5, label='Pareto front', edgecolors='white', linewidths=0.5)
plt.colorbar(sc, ax=ax, label='Voltage (V)')
ax.set_xlabel('Corrosion loss (%)')
ax.set_ylabel('Net energy density (Wh/kg)')
ax.set_title('Design space — 2000 alloy configs at 25°C')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('alloy_design_space.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Calibration with your own data
If you have experimental discharge data, calibrate the model to your cell.

In [ ]:
from al_air_calibrate import calibrate, CALIB_CONFIG

# Replace with your own measurements:
# Conditions: 4M KOH, 25°C, paste cell
# Format: j_mA_cm2, V_cell
my_data = {
    'j_mA_cm2': np.array([2.0,  5.0,  10.0, 25.0, 50.0]),
    'V_cell':   np.array([1.35, 1.28, 1.20, 1.09, 0.97]),  # <-- replace with real values
    'source':   'my_paste_cell'
}

print('CALIB_CONFIG:', CALIB_CONFIG)
print('Running calibration...')

cal_params, rmse_before, rmse_after = calibrate(my_data, verbose=True)

print(f'\nRMSE before: {rmse_before:.1f} mV')
print(f'RMSE after:  {rmse_after:.1f} mV')
print(f'Target:      < 30 mV')
print(f'Status:      {"✓ Ready for publication" if rmse_after < 30 else "○ Needs more data points"}')

In [ ]:
# Plot calibration result
j_fine = np.linspace(0.5, 65, 80)
V_before = [cell_model(**CALIB_CONFIG, j_mA_cm2=j)['V_cell'] for j in j_fine]
V_after  = [cell_model(**CALIB_CONFIG, j_mA_cm2=j, params_override=cal_params)['V_cell'] for j in j_fine]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(j_fine, V_before, 'r--', alpha=0.7, label=f'Before (RMSE={rmse_before:.0f} mV)')
ax.plot(j_fine, V_after,  'g-',  linewidth=2, label=f'After  (RMSE={rmse_after:.0f} mV)')
ax.scatter(my_data['j_mA_cm2'], my_data['V_cell'], 
           color='white', s=60, zorder=5, label='Experimental', edgecolors='gray')
ax.set_xlabel('Current density (mA/cm²)')
ax.set_ylabel('Cell voltage (V)')
ax.set_title('Model calibration — paste cell data')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_result.png', dpi=150, bbox_inches='tight')
plt.show()

## Key findings summary

| Finding | Value | Notes |
|---|---|---|
| Optimal particle size | **~187 µm** | Paste-specific — not in solid-plate models |
| Optimal KOH | **~3.2 M** at 100µm, **~4.9 M** at 250µm | Coupled to particle size |
| Optimal temperature | **60–65 °C** for performance | 25°C is standard test condition |
| Corrosion crossover | **~28 mA/cm²** | Below: corrosion-dominated; above: kinetics |
| Mg in alloy | **Beneficial at 25°C, harmful above 55°C** | Temperature-dependent strategy |
| System energy | **884 Wh/kg** at 60°C · **753 Wh/kg** at 25°C | 3.5× / 3.0× Li-ion |

See [README.md](README.md) for all 16 findings and full citations.

Live app: [al-air-battery-lab.onrender.com](https://al-air-battery-lab.onrender.com)